# Notebook 1: Pipeline MIDAGRI (2020-2025)

**Objetivo**: cargar archivos `c-18` mensuales de MIDAGRI, manejar meses faltantes, recuperar Dic-2020 desde Dic-2021, calcular producción mensual real y exportar dataset limpio en formato largo.

**Decisiones clave**:
- Meses faltantes en el medio (Mayo-2021, Marzo-2022) → NaN explícito (no se inventa data)
- Negativos del `diff()` (revisiones retroactivas de MIDAGRI) → NaN
- Solo se conservan las 6 regiones objetivo
- Todos los cultivos pasan al merge integrado (filtrado Pareto-80 en `03_build_dataset_integrado.ipynb`)
- Output: formato largo (`region`, `cultivo`, `año`, `mes`, `produccion_mensual`)

### Qué hace esta celda

Importa librerías (`pandas`, `numpy`, `pathlib`) y configura opciones de visualización.

In [1]:
# ====================================================================
# CELDA 1: Importaciones y configuración de pandas
# ====================================================================
# Primer bloque ejecutable del pipeline MIDAGRI (notebook 01).
# Carga librerías para leer Excel, normalizar texto y manejar rutas.
# No produce archivos; prepara el entorno para las celdas siguientes.
# Salida implícita: módulos importados y opciones de visualización activas.

# Importación de módulos necesarios para esta celda.
# Se agrupan al inicio según convención PEP 8.
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 20)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\USER\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## 1. Configuración

Las rutas se resuelven automáticamente desde la raíz del repo (`BDS/` y `OUTPUTS/`). Si ejecutas desde otra carpeta, asegúrate de que `cwd` sea `SCRIPTS/` o la raíz `DM_TF/`.

### Qué hace esta celda

Define `ROOT`, rutas a `BDS/` y `OUTPUTS/`, años 2020–2025 y las 6 regiones objetivo.

In [2]:
# ====================================================================
# CELDA 2: Rutas BDS/OUTPUTS, años y regiones objetivo
# ====================================================================
# Define dónde están los Excel MIDAGRI (BDS/) y dónde escribir CSV (OUTPUTS/).
# Detecta la raíz del repo aunque el kernel se lance desde SCRIPTS/ o notebooks/.
# Fija el universo temporal (2020-2025) y las 6 regiones del estudio.
# Incluye diccionarios de meses y mapeo de nombres de cultivos rotos en Excel.

# Raíz del repositorio; se ajusta si cwd es notebooks/ o SCRIPTS/.
# Valor fijado aquí para reutilizarse en celdas posteriores.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    # Raíz del repositorio; se ajusta si cwd es notebooks/ o SCRIPTS/.
    # Valor fijado aquí para reutilizarse en celdas posteriores.
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    # Raíz del repositorio; se ajusta si cwd es notebooks/ o SCRIPTS/.
    # Valor fijado aquí para reutilizarse en celdas posteriores.
    ROOT = ROOT.parent

# Carpeta BDS/ con Excel MIDAGRI organizados por año.
# Valor fijado aquí para reutilizarse en celdas posteriores.
RUTA_BASE = ROOT / "BDS"          # Excel MIDAGRI (YYYY/MES.xlsx)
# Carpeta OUTPUTS/ donde se escriben CSV e figuras.
# Valor fijado aquí para reutilizarse en celdas posteriores.
RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)

# Ventana temporal principal del estudio agroclimático.
# Valor fijado aquí para reutilizarse en celdas posteriores.
ANIOS = [2020, 2021, 2022, 2023, 2024, 2025]

# Seis regiones con mayor peso agrícola en el proyecto.
# Valor fijado aquí para reutilizarse en celdas posteriores.
REGIONES_OBJETIVO = ['Piura', 'La Libertad', 'Ica', 'San Martin', 'Junin', 'Puno']

# Traduce nombre de mes en MAYÚSCULAS (Excel) a entero 1-12.
# Valor fijado aquí para reutilizarse en celdas posteriores.
MES_NUM = {
    'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6,
    'JULIO': 7, 'AGOSTO': 8, 'SEPTIEMBRE': 9, 'SETIEMBRE': 9,
    'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12
}
# Traduce entero de mes a nombre legible en español.
# Valor fijado aquí para reutilizarse en celdas posteriores.
NUM_MES = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio',
           7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}

# Fusiona variantes rotas de nombres de cultivos en MIDAGRI.
# Valor fijado aquí para reutilizarse en celdas posteriores.
MAPEO_CANONICO = {
    'arveja_seca': 'arveja_grano_seco',
    'haba_seca': 'haba_grano_seco',
    'espa_rrago': 'esparrago',
    'alca_chofa': 'alcachofa',
    'zana_horia': 'zanahoria',
    'pallar_gr_seco': 'pallar_grano_seco',
    'arveja_gr_seco': 'arveja_grano_seco',
    'haba_gr_seco': 'haba_grano_seco',
    'maiz_a_duro': 'maiz_amarillo_duro',
    'acei_tuna': 'aceituna'
}

# Salida de consola para trazabilidad y verificación intermedia.
print(f"Ruta base: {RUTA_BASE}")
print(f"Ruta output: {RUTA_OUTPUT}")
print(f"Anios: {ANIOS}")
print(f"Regiones: {REGIONES_OBJETIVO}")


Ruta base: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\BDS
Ruta output: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS
Anios: [2020, 2021, 2022, 2023, 2024, 2025]
Regiones: ['Piura', 'La Libertad', 'Ica', 'San Martin', 'Junin', 'Puno']


## 2. Funciones de utilidad

### Qué hace esta celda

Define funciones de normalización de nombres de cultivos y regiones (quita tildes, unifica variantes).

In [3]:
# ====================================================================
# CELDA 3: Funciones de normalización de nombres
# ====================================================================
# Utilidades reutilizadas en todo el notebook para homogeneizar etiquetas.
# MIDAGRI usa tildes, mayúsculas y guiones irregulares en columnas y regiones.
# Estas funciones convierten texto a snake_case ASCII y unifican variantes.
# No modifican datos aún; solo definen la lógica de limpieza nominal.

# Definición de `normalizar_nombre()`.
# Convierte texto a snake_case ASCII sin tildes ni símbolos.
def normalizar_nombre(s):
    """Quita tildes, lowercase, reemplaza no-alfanumerico con _"""
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', errors='ignore').decode()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

# Definición de `aplicar_canonico()`.
# Busca alias en MAPEO_CANONICO; si no existe, deja el nombre.
def aplicar_canonico(col):
    """Aplica el mapeo canonico si existe"""
    return MAPEO_CANONICO.get(col, col)

# Definición de `normalizar_region()`.
# Quita tildes de nombres de departamento para comparar con MIDAGRI.
def normalizar_region(s):
    """Quita tildes de nombres de regiones"""
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', errors='ignore').decode()

# Definición de `extraer_mes_de_filename()`.
# Infiere mes 1-12 desde el stem del archivo Excel.
def extraer_mes_de_filename(filename):
    """Extrae mes (1-12) del nombre del archivo. Tolera espacios y variaciones."""
    name_norm = Path(filename).stem.upper()
    name_norm = re.sub(r'\s+', ' ', name_norm).strip()
    # Bucle de iteración sobre elementos de una colección.
    for mes_str, num in MES_NUM.items():
        if mes_str in name_norm:
            return num
    return None


## 3. Descubrimiento de archivos y validación de meses faltantes

Recorre las carpetas de cada año y reporta qué meses están presentes y cuáles faltan.

**Esperado** (basado en lo que sabemos del dataset):
- 2020: falta diciembre
- 2021: falta mayo
- 2022: falta marzo
- 2023, 2024, 2025: completos

### Qué hace esta celda

Implementa `descubrir_archivos`: escanea carpetas anuales y reporta meses presentes/faltantes en Excel c-18.

In [4]:
# ====================================================================
# CELDA 4: Descubrimiento de archivos Excel c-18
# ====================================================================
# Escanea BDS/YYYY/*.xlsx y construye un índice (año, mes) → ruta.
# El mes se infiere del nombre del archivo (p. ej. 'MARZO 2021.xlsx').
# Genera un reporte de cobertura: meses presentes vs. faltantes por año.
# Salida: dict `archivos` usado por todas las cargas posteriores.

# Definición de `descubrir_archivos()`.
# Indexa todos los .xlsx por (año, mes) bajo BDS/YYYY/.
def descubrir_archivos(ruta_base, anios):
    """Recorre subcarpetas de cada anio y mapea (anio, mes) -> ruta del archivo."""
    archivos = {}
    # Iteración temporal para reportes de cobertura o auditoría.
    for anio in anios:
        carpeta = ruta_base / str(anio)
        if not carpeta.exists():
            # Salida de consola para trazabilidad y verificación intermedia.
            print(f"[!] Carpeta no existe: {carpeta}")
            continue
        # Bucle de iteración sobre elementos de una colección.
        for archivo in carpeta.glob("*.xlsx"):
            mes = extraer_mes_de_filename(archivo.name)
            if mes is None:
                # Salida de consola para trazabilidad y verificación intermedia.
                print(f"[!] No pude extraer mes de: {archivo.name}")
                continue
            archivos[(anio, mes)] = archivo
    return archivos

archivos = descubrir_archivos(RUTA_BASE, ANIOS)

# Salida de consola para trazabilidad y verificación intermedia.
print("\n=== REPORTE DE COBERTURA ===\n")
print(f"{'Anio':<6} {'Presentes':<11} {'Faltantes'}")
print("-" * 60)
faltantes_global = []
# Iteración temporal para reportes de cobertura o auditoría.
for anio in ANIOS:
    meses_presentes = sorted([m for (a, m) in archivos if a == anio])
    meses_faltantes = sorted(set(range(1, 13)) - set(meses_presentes))
    nombres_faltantes = [NUM_MES[m] for m in meses_faltantes]
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"{anio:<6} {len(meses_presentes):>3}/12      {nombres_faltantes if nombres_faltantes else 'Completo'}")
    # Bucle de iteración sobre elementos de una colección.
    for m in meses_faltantes:
        faltantes_global.append((anio, m))

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\nTotal archivos descubiertos: {len(archivos)}")
print(f"Total meses faltantes: {len(faltantes_global)}")



=== REPORTE DE COBERTURA ===

Anio   Presentes   Faltantes
------------------------------------------------------------
2020    11/12      ['Diciembre']
2021    11/12      ['Mayo']
2022    11/12      ['Marzo']
2023    12/12      Completo
2024    12/12      Completo
2025    12/12      Completo

Total archivos descubiertos: 69
Total meses faltantes: 3


## 4. Carga y uniformización de archivos

Cada archivo c-18 tiene 5 bloques de columnas pegados horizontalmente. La función:
1. Detecta dinámicamente la fila del header (busca "Años")
2. Carga, hace ffill de regiones, elimina "Total Nacional"
3. Quita columnas espejo (`Región.1, Años.1, ...`)
4. Normaliza nombres de cultivos y aplica mapeo canónico
5. Combina columnas con el mismo nombre canónico (para reconciliar variantes como `'espárrago'` y `'espá- rrago'`)
6. Filtra al año del archivo y a las 6 regiones objetivo

### Qué hace esta celda

Implementa `cargar_archivo`: lee hoja c-18, detecta header, limpia bloques duplicados y filtra por año del archivo.

In [5]:
# ====================================================================
# CELDA 5: Carga y limpieza de cada archivo MIDAGRI
# ====================================================================
# Lee la hoja c-18 de cada Excel: producción acumulada por región y cultivo.
# Detecta la fila de encabezado, elimina totales nacionales y columnas espejo.
# Normaliza nombres, filtra regiones objetivo y concatena todos los meses.
# Salida: `df_acumulado` en formato ancho (una columna por cultivo).

# Definición de `cargar_archivo()`.
# Pipeline completo de lectura y limpieza de un Excel c-18.
def cargar_archivo(ruta, anio_archivo, mes_archivo, hoja='c-18'):
    """Carga un archivo MIDAGRI c-18 y devuelve un DataFrame limpio en formato ancho."""
    # 1. Detectar fila del header
    # Lectura de archivo tabular desde disco hacia un DataFrame pandas.
    df_temp = pd.read_excel(ruta, sheet_name=hoja, header=None)
    fila_header = None
    # Bucle de iteración sobre elementos de una colección.
    for i, row in df_temp.iterrows():
        if row.astype(str).str.contains('Anos|Años', case=False, na=False, regex=True).any():
            fila_header = i
            break
    if fila_header is None:
        raise ValueError(f"No se encontro header con 'Anos' en {ruta}")
    
    # Lectura de archivo tabular desde disco hacia un DataFrame pandas.
    df = pd.read_excel(ruta, sheet_name=hoja, skiprows=fila_header)
    df = df.dropna(how='all').reset_index(drop=True)
    
    # 2. ffill regiones, eliminar Total Nacional
    df['Región'] = df['Región'].ffill()
    df = df[df['Región'] != 'Total Nacional']
    
    # 3. Extraer anio
    df['Años'] = df['Años'].astype(str).str.extract(r'(\d{4})')[0].astype(float)
    df = df.dropna(subset=['Años'])
    df['Años'] = df['Años'].astype(int)
    
    # 4. Eliminar columnas espejo (Region.1, Anos.1, etc.)
    cols_espejo = [c for c in df.columns if re.match(r'(Región|Años)\.\d+', str(c))]
    df = df.drop(columns=cols_espejo)
    
    # 5. Uniformizar nombres
    rename_map = {}
    # Bucle de iteración sobre elementos de una colección.
    for col in df.columns:
        if col in ('Región', 'Años'):
            rename_map[col] = 'region' if col == 'Región' else 'anio'
        else:
            norm = normalizar_nombre(col)
            rename_map[col] = aplicar_canonico(norm)
    df = df.rename(columns=rename_map)
    
    # 6. Combinar columnas duplicadas (mismo nombre canonico)
    if df.columns.duplicated().any():
        df = df.groupby(level=0, axis=1).apply(lambda x: x.bfill(axis=1).iloc[:, 0])
    
    # 7. Normalizar nombres de regiones
    df['region'] = df['region'].apply(normalizar_region)
    
    # 8. Filtrar regiones y anio
    df = df[df['region'].isin(REGIONES_OBJETIVO)]
    df = df[df['anio'] == anio_archivo]
    df['mes_num'] = mes_archivo
    
    # 9. Convertir cultivos a numerico
    cols_meta = ['region', 'anio', 'mes_num']
    cols_cultivos = [c for c in df.columns if c not in cols_meta]
    # Bucle columna a columna sobre variables de cultivo o clima.
    # Permite aplicar la misma transformación a cada variable homogéneamente.
    for c in cols_cultivos:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    return df.reset_index(drop=True)

dfs = []
# Recorre cronológicamente todos los Excel descubiertos.
# Cada par (año, mes) se carga y apendiza a la lista dfs.
for (anio, mes), ruta in sorted(archivos.items()):
    df = cargar_archivo(ruta, anio, mes)
    dfs.append(df)

# Concatena verticalmente DataFrames con la misstructura de columnas.
df_acumulado = pd.concat(dfs, ignore_index=True)
# Salida de consola para trazabilidad y verificación intermedia.
print(f"Dataset acumulado cargado: {df_acumulado.shape}")
print(f"Cultivos detectados: {df_acumulado.shape[1] - 3}")
print(f"\nMuestra:")
print(df_acumulado[['region','anio','mes_num']].head(10))


Dataset acumulado cargado: (414, 57)
Cultivos detectados: 54

Muestra:
        region  anio  mes_num
0          Ica  2020        1
1        Junin  2020        1
2  La Libertad  2020        1
3        Piura  2020        1
4         Puno  2020        1
5   San Martin  2020        1
6          Ica  2020        2
7        Junin  2020        2
8  La Libertad  2020        2
9        Piura  2020        2


## 5. Recuperación de Diciembre 2020

El archivo de Diciembre 2021 contiene una fila "2020" que reporta el **total anual completo de 2020** (porque su periodo es "Enero-Diciembre 2020-2021"). Ese total = acumulado a Dic-2020.

Lo extraemos y lo agregamos como `(anio=2020, mes_num=12)`.

### Qué hace esta celda

Implementa `recuperar_dic_2020`: extrae diciembre 2020 desde el Excel de diciembre 2021 (fila de total anual 2020).

In [6]:
# ====================================================================
# CELDA 6: Recuperación de diciembre 2020 desde dic-2021
# ====================================================================
# MIDAGRI no publicó dic-2020; el Excel de dic-2021 incluye filas comparativas 2020.
# Extrae esas filas como proxy del acumulado de diciembre 2020.
# Evita duplicar filas si ya existían registros para (2020, 12).
# Salida: `df_acumulado` actualizado con diciembre 2020 recuperado.

# Definición de `recuperar_dic_2020()`.
# Imputa dic-2020 usando filas 2020 del Excel dic-2021.
def recuperar_dic_2020(archivos, df_acumulado):
    """Si existe archivo dic-2021, extrae fila 2020 (total anual) como dic-2020 acumulado."""
    if (2021, 12) not in archivos:
        # Salida de consola para trazabilidad y verificación intermedia.
        print("[!] No se encontro archivo de Dic 2021. Diciembre 2020 quedara como NaN.")
        return df_acumulado
    
    ruta_dic21 = archivos[(2021, 12)]
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"Recuperando Dic-2020 desde: {ruta_dic21.name}")
    
    # Cargar el archivo dic-2021 pero filtrando solo las filas de "2020" (el total anual comparativo)
    df_dic20 = cargar_archivo(ruta_dic21, anio_archivo=2020, mes_archivo=12)
    
    # Verificar si ya existe una fila (2020, 12) en df_acumulado (improbable, pero por seguridad)
    mask_existente = (df_acumulado['anio'] == 2020) & (df_acumulado['mes_num'] == 12)
    if mask_existente.any():
        # Salida de consola para trazabilidad y verificación intermedia.
        print("[!] Ya existian filas para Dic 2020. Se reemplazan con las recuperadas.")
        # DataFrame acumulado MIDAGRI (producción YTD por mes).
        df_acumulado = df_acumulado[~mask_existente]
    
    # Concatena verticalmente DataFrames con la misstructura de columnas.
    df_resultado = pd.concat([df_acumulado, df_dic20], ignore_index=True)
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"Filas agregadas: {len(df_dic20)} (una por cada region objetivo)")
    return df_resultado

# DataFrame acumulado MIDAGRI (producción YTD por mes).
df_acumulado = recuperar_dic_2020(archivos, df_acumulado)
# Salida de consola para trazabilidad y verificación intermedia.
print(f"\nDataset tras recuperacion Dic-2020: {df_acumulado.shape}")


Recuperando Dic-2020 desde: DICIEMBRE.xlsx
Filas agregadas: 6 (una por cada region objetivo)

Dataset tras recuperacion Dic-2020: (420, 57)


## 6. Construcción del grid completo (inserción de NaN para meses faltantes)

Esto es **crítico**: cuando falta un mes en el medio (mayo 2021, marzo 2022), el `diff()` ingenuo inflaría el mes siguiente. Insertando filas NaN explícitas, el `diff()` devolverá NaN y el dato faltante se mantendrá visible y honesto.

### Qué hace esta celda

Construye grid completo región×año×mes e inserta NaN donde MIDAGRI no publicó boletín (evita diff inflado).

In [7]:
# ====================================================================
# CELDA 7: Grid completo región×año×mes con NaN explícitos
# ====================================================================
# Construye el producto cartesiano región × año × mes (1-12).
# Hace left merge con datos reales: meses sin archivo quedan como NaN.
# Documenta explícitamente los huecos temporales (no se imputan valores).
# Salida: grilla completa ordenada, base para calcular producción mensual.

# Definición de `construir_grid_completo()`.
# Outer grid región×año×mes; huecos quedan como NaN.
def construir_grid_completo(df, regiones, anios):
    """Crea grid (region x anio x mes) y hace left merge. Filas faltantes quedan NaN."""
    grid = pd.MultiIndex.from_product(
        [regiones, anios, range(1, 13)],
        names=['region', 'anio', 'mes_num']
    ).to_frame(index=False)
    # Operación merge (join) entre DataFrames.
    # Une claves compartidas; how='left' preserva filas del lado izquierdo.
    return grid.merge(df, on=['region', 'anio', 'mes_num'], how='left')

# DataFrame acumulado MIDAGRI (producción YTD por mes).
df_acumulado = construir_grid_completo(df_acumulado, REGIONES_OBJETIVO, ANIOS)
# DataFrame acumulado MIDAGRI (producción YTD por mes).
df_acumulado = df_acumulado.sort_values(['region','anio','mes_num']).reset_index(drop=True)

# Salida de consola para trazabilidad y verificación intermedia.
print(f"Grid completo: {df_acumulado.shape}")
print(f"Esperado: {len(REGIONES_OBJETIVO)} regiones x {len(ANIOS)} anios x 12 meses = {len(REGIONES_OBJETIVO)*len(ANIOS)*12}")

cols_cultivos = [c for c in df_acumulado.columns if c not in ('region','anio','mes_num')]
mask_vacio = df_acumulado[cols_cultivos].isna().all(axis=1)
# Salida de consola para trazabilidad y verificación intermedia.
print(f"\nFilas con todos los cultivos NaN (= meses no disponibles): {mask_vacio.sum()}")

# Lista meses donde ningún cultivo tiene dato (archivo faltante).
if mask_vacio.any():
    filas_vacias = df_acumulado[mask_vacio][['region','anio','mes_num']]
    meses_vacios_unicos = filas_vacias[['anio','mes_num']].drop_duplicates().sort_values(['anio','mes_num'])
    # Salida de consola para trazabilidad y verificación intermedia.
    print("Meses afectados (uno por region):")
    # Bucle de iteración sobre elementos de una colección.
    for _, r in meses_vacios_unicos.iterrows():
        # Salida de consola para trazabilidad y verificación intermedia.
        print(f"  {int(r['anio'])}-{NUM_MES[int(r['mes_num'])]}")


Grid completo: (432, 57)
Esperado: 6 regiones x 6 anios x 12 meses = 432

Filas con todos los cultivos NaN (= meses no disponibles): 12
Meses afectados (uno por region):
  2021-Mayo
  2022-Marzo


## 7. Cálculo de producción mensual real

Aplica `diff()` por (region, año) sobre cada cultivo. Tres casos importantes:

| Caso | Comportamiento |
|---|---|
| Mes 1 (enero) presente | Conserva valor original (es enero real, primer acumulado del año) |
| Mes con anterior faltante | NaN (correcto: no se puede calcular el delta) |
| Diff negativo (revisión retroactiva) | Se convierte a NaN |

### Qué hace esta celda

Aplica `diff()` por región y año para obtener producción mensual real; enero conserva acumulado; negativos → NaN.

In [8]:
# ====================================================================
# CELDA 8: Producción mensual real vía diff() del acumulado
# ====================================================================
# MIDAGRI reporta producción acumulada; la mensual = diff mes a mes por región/año.
# Enero conserva el valor original (no hay mes anterior en la serie anual).
# Negativos del diff (revisiones retroactivas) se convierten a NaN.
# Salida: `df_mensual` con producción mensual estimada por cultivo.

cols_cultivos = [c for c in df_acumulado.columns if c not in ('region','anio','mes_num')]

# DataFrame acumulado MIDAGRI (producción YTD por mes).
df_mensual = df_acumulado.copy()
# Bucle columna a columna sobre variables de cultivo o clima.
# Permite aplicar la misma transformación a cada variable homogéneamente.
for col in cols_cultivos:
    serie_diff = df_acumulado.groupby(['region','anio'])[col].diff()
    # Para enero (mes_num=1): conservar valor original (es enero real)
    es_enero = df_acumulado['mes_num'] == 1
    # DataFrame con producción mensual (diff del acumulado).
    df_mensual[col] = serie_diff
    # DataFrame acumulado MIDAGRI (producción YTD por mes).
    df_mensual.loc[es_enero, col] = df_acumulado.loc[es_enero, col]

neg_count = 0
# Bucle columna a columna sobre variables de cultivo o clima.
# Permite aplicar la misma transformación a cada variable homogéneamente.
for col in cols_cultivos:
    mask_neg = df_mensual[col] < 0
    neg_count += mask_neg.sum()
    # DataFrame con producción mensual (diff del acumulado).
    df_mensual.loc[mask_neg, col] = np.nan

# Salida de consola para trazabilidad y verificación intermedia.
print(f"Produccion mensual calculada.")
print(f"Negativos convertidos a NaN: {neg_count} (revisiones retroactivas de MIDAGRI)")

total_checks = 0
violaciones = 0
# Bucle columna a columna sobre variables de cultivo o clima.
# Permite aplicar la misma transformación a cada variable homogéneamente.
for col in cols_cultivos:
    # Iteración por subgrupos del DataFrame (groupby).
    # Cada iteración procesa un bloque homogéneo (región, cultivo, distrito, etc.).
    for (region, anio), grupo in df_acumulado.groupby(['region','anio']):
        valores = grupo.sort_values('mes_num')[col].dropna().tolist()
        # Bucle de iteración sobre elementos de una colección.
        for i in range(1, len(valores)):
            total_checks += 1
            if valores[i] < valores[i-1]:
                violaciones += 1

if total_checks > 0:
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"\nMonotonia del acumulado: {100*(1-violaciones/total_checks):.2f}% ({violaciones} violaciones de {total_checks})")
    print("(violaciones son normales: revisiones retroactivas de datos preliminares)")


Produccion mensual calculada.
Negativos convertidos a NaN: 257 (revisiones retroactivas de MIDAGRI)

Monotonia del acumulado: 98.71% (268 violaciones de 20736)
(violaciones son normales: revisiones retroactivas de datos preliminares)


# 8 Mapeo original de cultivos mal escritos

### Qué hace esta celda

Diagnóstico de variantes de nombres de cultivos en archivos crudos (antes del mapeo canónico).

In [9]:
# ====================================================================
# CELDA 9: Diagnóstico de variantes de nombres de cultivos
# ====================================================================
# Celda exploratoria: detecta columnas que representan el mismo cultivo.
# Carga archivos SIN mapeo canónico para ver nombres crudos del Excel.
# Agrupa variantes (p. ej. acei_tuna vs aceituna) y sugiere MAPEO_CANONICO.
# No altera el pipeline principal; sirve para auditar la limpieza nominal.

print("=" * 70)
print("DIAGNÓSTICO: Detectando variantes de nombres de cultivos")
print("=" * 70)


# Definición de `cargar_archivo_crudo()`.
# Igual que cargar_archivo pero SIN mapeo canónico (diagnóstico).
def cargar_archivo_crudo(ruta, anio_archivo, mes_archivo, hoja='c-18'):
    """Versión de cargar_archivo() que NO aplica MAPEO_CANONICO (para diagnóstico)"""
    # 1. Detectar fila del header
    # Lectura de archivo tabular desde disco hacia un DataFrame pandas.
    df_temp = pd.read_excel(ruta, sheet_name=hoja, header=None)
    fila_header = None
    # Bucle de iteración sobre elementos de una colección.
    for i, row in df_temp.iterrows():
        if row.astype(str).str.contains('Anos|Años', case=False, na=False, regex=True).any():
            fila_header = i
            break
    if fila_header is None:
        raise ValueError(f"No se encontro header con 'Anos' en {ruta}")
    
    # Lectura de archivo tabular desde disco hacia un DataFrame pandas.
    df = pd.read_excel(ruta, sheet_name=hoja, skiprows=fila_header)
    df = df.dropna(how='all').reset_index(drop=True)
    
    # 2. ffill regiones, eliminar Total Nacional
    df['Región'] = df['Región'].ffill()
    df = df[df['Región'] != 'Total Nacional']
    
    # 3. Extraer anio
    df['Años'] = df['Años'].astype(str).str.extract(r'(\d{4})')[0].astype(float)
    df = df.dropna(subset=['Años'])
    df['Años'] = df['Años'].astype(int)
    
    # 4. Eliminar columnas espejo
    cols_espejo = [c for c in df.columns if re.match(r'(Región|Años)\.\d+', str(c))]
    df = df.drop(columns=cols_espejo)
    
    # 5. Normalizar nombres SIN aplicar mapeo canónico
    rename_map = {}
    # Bucle de iteración sobre elementos de una colección.
    for col in df.columns:
        if col in ('Región', 'Años'):
            rename_map[col] = 'region' if col == 'Región' else 'anio'
        else:
            norm = normalizar_nombre(col)
            rename_map[col] = norm  # ← NO aplicamos aplicar_canonico() aquí
    df = df.rename(columns=rename_map)
    
    # 6. Combinar columnas duplicadas
    if df.columns.duplicated().any():
        df = df.groupby(level=0, axis=1).apply(lambda x: x.bfill(axis=1).iloc[:, 0])
    
    return df

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\nCargando archivos SIN mapeo canónico para diagnóstico...\n")

cultivos_crudos = set()
# Recorre cronológicamente todos los Excel descubiertos.
# Cada par (año, mes) se carga y apendiza a la lista dfs.
for (anio, mes), ruta in sorted(archivos.items()):  # Muestreo de 5 archivos
    df_crudo = cargar_archivo_crudo(ruta, anio, mes)
    cols_cultivos_crudo = [c for c in df_crudo.columns if c not in ('region', 'anio')]
    cultivos_crudos.update(cols_cultivos_crudo)
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"  {anio}-{NUM_MES[mes]}: {len(cols_cultivos_crudo)} cultivos")

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\nTotal cultivos únicos (sin mapeo): {len(cultivos_crudos)}\n")

# Importación de módulos necesarios para esta celda.
# Se agrupan al inicio según convención PEP 8.
from collections import defaultdict

grupos = defaultdict(list)
# Agrupa nombres de cultivos que difieren solo por guiones o typos.
for c in cultivos_crudos:
    clave = re.sub(r'_', '', c)  # ej: 'acei_tuna' -> 'aceituna'
    grupos[clave].append(c)

# Salida de consola para trazabilidad y verificación intermedia.
print("=" * 70)
print("VARIANTES DETECTADAS (grupos con >1 nombre)")
print("=" * 70)

sugerencias = {}
variantes_encontradas = False

# Iteración por subgrupos del DataFrame (groupby).
# Cada iteración procesa un bloque homogéneo (región, cultivo, distrito, etc.).
for clave, variantes in sorted(grupos.items()):
    if len(variantes) > 1:
        variantes_encontradas = True
        # La canónica suele ser la más corta o la "natural" sin underscores extra
        canonica = min(variantes, key=lambda x: x.count('_'))
        otras = [v for v in variantes if v != canonica]
        # Salida de consola para trazabilidad y verificación intermedia.
        print(f"\n✓ Grupo: {clave}")
        print(f"  → Canónica sugerida: '{canonica}'")
        # Bucle de iteración sobre elementos de una colección.
        for v in otras:
            # Salida de consola para trazabilidad y verificación intermedia.
            print(f"  → Variante: '{v}'")
            sugerencias[v] = canonica

if not variantes_encontradas:
    # Salida de consola para trazabilidad y verificación intermedia.
    print("\n✓ No se encontraron variantes. Los datos están limpios.")
else:
    # Salida de consola para trazabilidad y verificación intermedia.
    print("\n" + "=" * 70)
    print("MAPEO SUGERIDO (copia esto a MAPEO_CANONICO si es necesario)")
    print("=" * 70)
    print("MAPEO_CANONICO = {")
    # Bucle de iteración sobre elementos de una colección.
    for variante, canonica in sorted(sugerencias.items()):
        # Salida de consola para trazabilidad y verificación intermedia.
        print(f"    '{variante}': '{canonica}',")
    print("}")

# Salida de consola para trazabilidad y verificación intermedia.
print("\n" + "=" * 70)
print("MAPEO ACTUAL en tu código:")
print("=" * 70)
# Bucle de iteración sobre elementos de una colección.
for k, v in MAPEO_CANONICO.items():
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"    '{k}': '{v}',")


DIAGNÓSTICO: Detectando variantes de nombres de cultivos

Cargando archivos SIN mapeo canónico para diagnóstico...

  2020-Enero: 54 cultivos
  2020-Febrero: 54 cultivos
  2020-Marzo: 54 cultivos
  2020-Abril: 54 cultivos
  2020-Mayo: 54 cultivos
  2020-Junio: 54 cultivos
  2020-Julio: 54 cultivos
  2020-Agosto: 54 cultivos
  2020-Septiembre: 54 cultivos
  2020-Octubre: 54 cultivos
  2020-Noviembre: 54 cultivos
  2021-Enero: 54 cultivos
  2021-Febrero: 54 cultivos
  2021-Marzo: 54 cultivos
  2021-Abril: 54 cultivos
  2021-Junio: 54 cultivos
  2021-Julio: 54 cultivos
  2021-Agosto: 54 cultivos
  2021-Septiembre: 54 cultivos
  2021-Octubre: 54 cultivos
  2021-Noviembre: 54 cultivos
  2021-Diciembre: 54 cultivos
  2022-Enero: 54 cultivos
  2022-Febrero: 54 cultivos
  2022-Abril: 54 cultivos
  2022-Mayo: 54 cultivos
  2022-Junio: 54 cultivos
  2022-Julio: 54 cultivos
  2022-Agosto: 54 cultivos
  2022-Septiembre: 54 cultivos
  2022-Octubre: 54 cultivos
  2022-Noviembre: 54 cultivos
  2022-D

## 8.5 Conversión a formato largo

Pasamos de una fila por (region, año, mes) con N columnas de cultivos, a una fila por (region, cultivo, año, mes) con la columna `produccion_mensual`.

### Qué hace esta celda

Convierte de formato ancho (columnas=cultivos) a largo (`region`, `cultivo`, `mes`, `produccion_mensual`).

In [10]:
# ====================================================================
# CELDA 10: Conversión a formato largo (melt)
# ====================================================================
# Pasa de formato ancho (columnas=cultivos) a largo (filas=observaciones).
# Cada fila queda: región, año, mes, cultivo, produccion_mensual.
# Añade mes_nombre legible y ordena para inspección y exportación.
# Salida: `df_largo`, formato estándar para merge con clima en notebook 03.

# Transforma de formato ancho (columnas=cultivos) a largo (filas=observaciones).
# id_vars conservan región, año y mes; value_vars son todos los cultivos.
df_largo = df_mensual.melt(
    id_vars=['region', 'anio', 'mes_num'],
    value_vars=cols_cultivos,
    var_name='cultivo',
    value_name='produccion_mensual'
)

# Dataset en formato largo listo para exportar o mergear.
df_largo['mes_nombre'] = df_largo['mes_num'].map(NUM_MES)

# Dataset en formato largo listo para exportar o mergear.
df_largo = df_largo[['region','anio','mes_num','mes_nombre','cultivo','produccion_mensual']]
# Dataset en formato largo listo para exportar o mergear.
df_largo = df_largo.sort_values(['region','anio','mes_num','cultivo']).reset_index(drop=True)

# Salida de consola para trazabilidad y verificación intermedia.
print(f"Dataset en formato largo: {df_largo.shape}")
print(f"\nResumen de NaN en produccion_mensual:")
print(f"  Total filas: {len(df_largo)}")
print(f"  Con valor: {df_largo['produccion_mensual'].notna().sum()}")
print(f"  NaN: {df_largo['produccion_mensual'].isna().sum()}")
print(f"\nMuestra:")
print(df_largo.head(10).to_string(index=False))


Dataset en formato largo: (23328, 6)

Resumen de NaN en produccion_mensual:
  Total filas: 23328
  Con valor: 21775
  NaN: 1553

Muestra:
region  anio  mes_num mes_nombre           cultivo  produccion_mensual
   Ica  2020        1      Enero          aceituna                0.00
   Ica  2020        1      Enero               aji              635.00
   Ica  2020        1      Enero               ajo               12.00
   Ica  2020        1      Enero         alcachofa                0.00
   Ica  2020        1      Enero           alfalfa            10684.89
   Ica  2020        1      Enero      algodon_rama             1432.79
   Ica  2020        1      Enero          arandano                0.00
   Ica  2020        1      Enero     arroz_cascara                0.00
   Ica  2020        1      Enero arveja_grano_seco                0.00
   Ica  2020        1      Enero      arveja_verde                0.00


### Qué hace esta celda

Segundo diagnóstico de variantes post-melt para detectar cultivos no mapeados.

In [11]:
# ====================================================================
# CELDA 11: Diagnóstico post-melt de cultivos
# ====================================================================
# Segunda pasada de diagnóstico sobre cultivos ya en formato largo.
# Verifica que el mapeo canónico no dejó duplicados semánticos.
# Cuenta filas con dato por variante para priorizar fusiones.
# Imprime sugerencias listas para pegar en MAPEO_CANONICO si hiciera falta.

# Importación de módulos necesarios para esta celda.
# Se agrupan al inicio según convención PEP 8.
import re
from collections import defaultdict

cultivos_unicos = sorted(df_largo['cultivo'].unique())
# Salida de consola para trazabilidad y verificación intermedia.
print(f"Total cultivos únicos: {len(cultivos_unicos)}\n")

grupos = defaultdict(list)
# Agrupa nombres de cultivos que difieren solo por guiones o typos.
for c in cultivos_unicos:
    clave = re.sub(r'_', '', c)  # ej: 'acei_tuna' -> 'aceituna'
    grupos[clave].append(c)

# Mostrar grupos con más de 1 variante (= variantes que deben fusionarse)
sugerencias = {}
# Iteración por subgrupos del DataFrame (groupby).
# Cada iteración procesa un bloque homogéneo (región, cultivo, distrito, etc.).
for clave, variantes in sorted(grupos.items()):
    if len(variantes) > 1:
        # La canónica suele ser la más corta o la "natural" sin underscores extra
        canonica = min(variantes, key=lambda x: x.count('_'))
        otras = [v for v in variantes if v != canonica]
        # Salida de consola para trazabilidad y verificación intermedia.
        print(f"  Canónica: '{canonica}'")
        # Bucle de iteración sobre elementos de una colección.
        for v in otras:
            # Salida de consola para trazabilidad y verificación intermedia.
            print(f"    Variante: '{v}'  → mapear a '{canonica}'")
            sugerencias[v] = canonica
        # Conteo de filas con datos en cada variante
        # Agrupa nombres de cultivos que difieren solo por guiones o typos.
        for v in variantes:
            n_con_dato = df_largo[(df_largo['cultivo']==v) & 
                                   df_largo['produccion_mensual'].notna()].shape[0]
            # Salida de consola para trazabilidad y verificación intermedia.
            print(f"      '{v}': {n_con_dato} filas con dato")
        print()

# Mostrar la lista lista para pegar en MAPEO_CANONICO
# Bucle de iteración sobre elementos de una colección.
for variante, canonica in sugerencias.items():
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"    '{variante}': '{canonica}',")


Total cultivos únicos: 54



## 9. Exportar

Genero el archivo principal:
- `midagri_largo.csv`: dataset en formato largo (para el merge agroclimático)

### Qué hace esta celda

Exporta `OUTPUTS/midagri_largo.csv` — insumo principal del notebook 03.

In [12]:
# ====================================================================
# CELDA 12: Exportar midagri_largo.csv
# ====================================================================
# Persiste el dataset limpio en OUTPUTS/midagri_largo.csv.
# Encoding utf-8-sig para compatibilidad con Excel en Windows.
# Este CSV es insumo obligatorio del notebook 03 (integración).
# Salida en disco: ~432 filas × 54 cultivos × 6 regiones (con NaN explícitos).

ruta_largo = RUTA_OUTPUT / "midagri_largo.csv"
# Exportación a CSV en OUTPUTS/ con encoding utf-8-sig (Excel-friendly).
# index=False evita escribir el índice numérico de pandas.
df_largo.to_csv(ruta_largo, index=False, encoding='utf-8-sig')
# Salida de consola para trazabilidad y verificación intermedia.
print(f"Exportado: {ruta_largo}")

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\n=== LISTO ===")
print(f"Dataset principal: {len(df_largo):,} filas")
print(f"Proximo paso: ejecutar notebook 02_nasa_pipeline.ipynb")


Exportado: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS\midagri_largo.csv

=== LISTO ===
Dataset principal: 23,328 filas
Proximo paso: ejecutar notebook 02_nasa_pipeline.ipynb


### Qué hace esta celda

Auditoría final: filas, regiones, cultivos únicos y conteo de NaN.

In [13]:
# ====================================================================
# CELDA 13: Auditoría final del dataset MIDAGRI
# ====================================================================
# Resumen descriptivo del CSV exportado antes de pasar al pipeline NASA.
# Cuenta NaN, ceros y valores positivos en produccion_mensual.
# Identifica cultivos con muchos ceros (no se siembran en ciertas regiones).
# Cierra el notebook 01 con métricas de calidad y cobertura regional.

print(f"Total filas: {len(df_largo):,}")
print(f"Total cultivos únicos: {df_largo['cultivo'].nunique()}")
print(f"Total regiones: {df_largo['region'].nunique()}")
print(f"Rango temporal: {df_largo['anio'].min()}-{df_largo['anio'].max()}")

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\n=== Distribución de produccion_mensual ===")
print(f"  NaN:       {df_largo['produccion_mensual'].isna().sum():>7,}  ({df_largo['produccion_mensual'].isna().mean()*100:>5.1f}%)")
print(f"  Igual a 0: {(df_largo['produccion_mensual']==0).sum():>7,}  ({(df_largo['produccion_mensual']==0).mean()*100:>5.1f}%)")
print(f"  > 0:       {(df_largo['produccion_mensual']>0).sum():>7,}  ({(df_largo['produccion_mensual']>0).mean()*100:>5.1f}%)")

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\n=== Top 10 cultivos con MÁS proporción de ceros ===")
print("(probablemente cultivos que no se producen en algunas regiones)")
ceros_por_cultivo = df_largo.groupby('cultivo')['produccion_mensual'].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).sort_values(ascending=False).head(10)
# Salida de consola para trazabilidad y verificación intermedia.
print(ceros_por_cultivo.round(1).to_string())

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\n=== Top 10 cultivos por producción mensual promedio (filas con valor) ===")
top_prod = df_largo[df_largo['produccion_mensual'] > 0].groupby('cultivo')['produccion_mensual'].mean().sort_values(ascending=False).head(10)
# Salida de consola para trazabilidad y verificación intermedia.
print(top_prod.round(1).to_string())

# Salida de consola para trazabilidad y verificación intermedia.
print(f"\n=== Cultivos con producción real (>0) por región ===")
# Bucle de iteración sobre elementos de una colección.
for region in sorted(df_largo['region'].unique()):
    sub = df_largo[df_largo['region'] == region]
    cultivos_activos = sub[sub['produccion_mensual'] > 0]['cultivo'].nunique()
    # Salida de consola para trazabilidad y verificación intermedia.
    print(f"  {region:<15}: {cultivos_activos}/54 cultivos con producción real")


Total filas: 23,328
Total cultivos únicos: 54
Total regiones: 6
Rango temporal: 2020-2025

=== Distribución de produccion_mensual ===
  NaN:         1,553  (  6.7%)
  Igual a 0:  11,018  ( 47.2%)
  > 0:        10,757  ( 46.1%)

=== Top 10 cultivos con MÁS proporción de ceros ===
(probablemente cultivos que no se producen en algunas regiones)
cultivo
aceituna             80.3
palma_aceitera       78.7
maiz_chala           78.0
pallar_grano_seco    76.4
piquillo             75.5
oregano              73.4
tangelo              71.8
tuna                 70.1
cebada_forrajera     69.7
algodon_rama         69.2

=== Top 10 cultivos por producción mensual promedio (filas con valor) ===
cultivo
cana_para_azucar    238735.3
avena_forrajera      88641.8
palma_aceitera       45092.7
alfalfa              40989.4
arroz_cascara        38028.7
papa                 37614.8
uva                  17451.0
cebada_forrajera     16771.5
maiz_chala           16138.8
platano              14957.4

=== Cultivos c